# Date Cleaning

Clean publication dates from the Goodreads raw data and create `first_publish_year` for era-based analysis.

Tasks covered:
- standardize mixed date formats
- extract first publication year
- flag impossible or unreliable values
- decide how missing years are handled


In [1]:
import re
from pathlib import Path

import pandas as pd

RAW_DATA_PATH = Path('../data/raw/books_goodreads.csv')
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = Path('data/raw/books_goodreads.csv')

df = pd.read_csv(RAW_DATA_PATH, dtype='string', keep_default_na=False)
df.shape


(52478, 25)

## Cleaning Decisions

- Use `firstPublishDate` as the primary source for `first_publish_year`.
- If `firstPublishDate` is missing, fall back to `publishDate` and mark the source as `publishDate_fallback`.
- Keep missing years as `NA`; do not impute them.
- Strip ordinal suffixes such as `1st`, `2nd`, `3rd`, and `4th` before parsing.
- For `M/D/YY` values, infer the century with project-aware rules:
  - prefer `20YY` only when it is within the 1997-2024 project window and not after the edition `publishDate` year when that year is available;
  - otherwise use `19YY`;
  - flag every two-digit-year parse as inferred because older classics can remain century-ambiguous.
- Treat years before 1450 or after 2024 as invalid for this project and set `first_publish_year` to `NA`.
- Add `in_project_scope_1997_2024` so modern youth-fiction analysis can filter cleanly.


In [2]:
PROJECT_START_YEAR = 1997
PROJECT_END_YEAR = 2024
MIN_VALID_PUBLICATION_YEAR = 1450

TWO_DIGIT_DATE_RE = re.compile(r'^(\d{1,2})/(\d{1,2})/(\d{2})$')
FOUR_DIGIT_DATE_RE = re.compile(r'^(\d{1,2})/(\d{1,2})/(\d{4})$')
YEAR_ONLY_RE = re.compile(r'^(\d{4})$')
YEAR_RE = re.compile(r'(\d{4})')

def normalize_date_text(value):
    value = str(value or '').strip()
    value = re.sub(r'(\d)(st|nd|rd|th)', r'\1', value)
    return re.sub(r'\s+', ' ', value)

def infer_two_digit_year(two_digit_year, edition_publish_year=None):
    yy = int(two_digit_year)
    year_2000 = 2000 + yy
    year_1900 = 1900 + yy

    can_be_modern = PROJECT_START_YEAR <= year_2000 <= PROJECT_END_YEAR
    if edition_publish_year is not None and pd.notna(edition_publish_year):
        can_be_modern = can_be_modern and year_2000 <= int(edition_publish_year) + 1

    if can_be_modern:
        return year_2000, 'two_digit_year_inferred_modern'
    return year_1900, 'two_digit_year_inferred_pre_2000'

def parse_publication_year(value, edition_publish_year=None):
    value = normalize_date_text(value)
    if not value:
        return pd.NA, 'missing_raw_date'

    two_digit_match = TWO_DIGIT_DATE_RE.match(value)
    if two_digit_match:
        year, issue = infer_two_digit_year(two_digit_match.group(3), edition_publish_year)
        return year, issue

    four_digit_match = FOUR_DIGIT_DATE_RE.match(value)
    if four_digit_match:
        return int(four_digit_match.group(3)), 'parsed_four_digit_slash_date'

    year_only_match = YEAR_ONLY_RE.match(value)
    if year_only_match:
        return int(year_only_match.group(1)), 'parsed_year_only'

    parsed = pd.to_datetime(pd.Series([value]), errors='coerce', format='mixed').iloc[0]
    if pd.notna(parsed):
        return int(parsed.year), 'parsed_text_date'

    year_match = YEAR_RE.search(value)
    if year_match:
        return int(year_match.group(1)), 'parsed_year_from_unrecognized_text'

    return pd.NA, 'unparseable_raw_date'


## Parse Edition Publication Year

`publishDate` is usually an edition/reprint date. It is not the final target, but it helps disambiguate two-digit years in `firstPublishDate`.

In [3]:
edition_year_results = df['publishDate'].apply(parse_publication_year)

df_dates = df.copy()
df_dates['publish_year_for_disambiguation'] = pd.array(
    [year for year, _ in edition_year_results],
    dtype='Int64',
)
df_dates['publish_year_parse_status'] = [status for _, status in edition_year_results]

df_dates[['publishDate', 'publish_year_for_disambiguation', 'publish_year_parse_status']].head(10)


,publishDate,publish_year_for_disambiguation,publish_year_parse_status
0,09/14/08,2008,two_digit_year_inferred_modern
1,09/28/04,2004,two_digit_year_inferred_modern
2,05/23/06,2006,two_digit_year_inferred_modern
3,10/10/00,2000,two_digit_year_inferred_modern
4,09/06/06,2006,two_digit_year_inferred_modern
5,03/14/06,2006,two_digit_year_inferred_modern
6,04/28/96,1996,two_digit_year_inferred_pre_2000
7,09/16/02,2002,two_digit_year_inferred_modern
8,09/25/12,2012,two_digit_year_inferred_modern
9,04/01/99,1999,two_digit_year_inferred_pre_2000


## Create `first_publish_year`

In [4]:
def choose_first_publish_year(row):
    first_raw = row['firstPublishDate']
    publish_raw = row['publishDate']
    edition_year = row['publish_year_for_disambiguation']

    if normalize_date_text(first_raw):
        year, parse_status = parse_publication_year(first_raw, edition_year)
        source = 'firstPublishDate'
    elif normalize_date_text(publish_raw):
        year, parse_status = parse_publication_year(publish_raw)
        source = 'publishDate_fallback'
        parse_status = f'missing_firstPublishDate__{parse_status}'
    else:
        year, parse_status, source = pd.NA, 'missing_both_date_fields', 'missing'

    invalid_reason = ''
    clean_year = year
    if pd.isna(year):
        invalid_reason = 'missing_or_unparseable_year'
    elif int(year) < MIN_VALID_PUBLICATION_YEAR:
        invalid_reason = 'before_min_valid_publication_year'
        clean_year = pd.NA
    elif int(year) > PROJECT_END_YEAR:
        invalid_reason = 'after_project_end_year'
        clean_year = pd.NA

    return pd.Series({
        'first_publish_year_raw_parsed': year,
        'first_publish_year': clean_year,
        'first_publish_year_source': source,
        'first_publish_year_parse_status': parse_status,
        'first_publish_year_invalid_reason': invalid_reason,
    })

first_year_columns = df_dates.apply(choose_first_publish_year, axis=1)
df_dates = pd.concat([df_dates, first_year_columns], axis=1)
df_dates['first_publish_year'] = pd.array(df_dates['first_publish_year'], dtype='Int64')
df_dates['first_publish_year_raw_parsed'] = pd.array(df_dates['first_publish_year_raw_parsed'], dtype='Int64')
df_dates['first_publish_year_missing'] = df_dates['first_publish_year'].isna()
df_dates['first_publish_year_uses_fallback'] = df_dates['first_publish_year_source'].eq('publishDate_fallback')
df_dates['first_publish_year_two_digit_inferred'] = df_dates['first_publish_year_parse_status'].str.contains('two_digit_year', regex=False)
df_dates['in_project_scope_1997_2024'] = df_dates['first_publish_year'].between(PROJECT_START_YEAR, PROJECT_END_YEAR)

df_dates[[
    'title',
    'publishDate',
    'firstPublishDate',
    'first_publish_year',
    'first_publish_year_source',
    'first_publish_year_parse_status',
    'first_publish_year_invalid_reason',
    'first_publish_year_two_digit_inferred',
    'in_project_scope_1997_2024',
]].head(20)


,title,publishDate,firstPublishDate,first_publish_year,first_publish_year_source,first_publish_year_parse_status,first_publish_year_invalid_reason,first_publish_year_two_digit_inferred,in_project_scope_1997_2024
0,The Hunger Games,09/14/08,,2008,publishDate_fallback,missing_firstPublishDate__two_digit_year_infer...,,True,True
1,Harry Potter and the Order of the Phoenix,09/28/04,06/21/03,2003,firstPublishDate,two_digit_year_inferred_modern,,True,True
2,To Kill a Mockingbird,05/23/06,07/11/60,1960,firstPublishDate,two_digit_year_inferred_pre_2000,,True,False
3,Pride and Prejudice,10/10/00,01/28/13,1913,firstPublishDate,two_digit_year_inferred_pre_2000,,True,False
4,Twilight,09/06/06,10/05/05,2005,firstPublishDate,two_digit_year_inferred_modern,,True,True
5,The Book Thief,03/14/06,09/01/05,2005,firstPublishDate,two_digit_year_inferred_modern,,True,True
6,Animal Farm,04/28/96,08/17/45,1945,firstPublishDate,two_digit_year_inferred_pre_2000,,True,False
7,The Chronicles of Narnia,09/16/02,10/28/56,1956,firstPublishDate,two_digit_year_inferred_pre_2000,,True,False
8,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,09/25/12,10/20/55,1955,firstPublishDate,two_digit_year_inferred_pre_2000,,True,False
9,Gone with the Wind,04/01/99,06/30/36,1936,firstPublishDate,two_digit_year_inferred_pre_2000,,True,False


## Date Cleaning Audit

In [7]:
date_cleaning_summary = pd.DataFrame({
    'metric': [
        'total_rows',
        'first_publish_year_present',
        'first_publish_year_missing_after_cleaning',
        'used_publishDate_fallback',
        'two_digit_year_inferred',
        'in_project_scope_1997_2024',
        'outside_project_scope_or_missing',
    ],
    'rows': [
        len(df_dates),
        int(df_dates['first_publish_year'].notna().sum()),
        int(df_dates['first_publish_year'].isna().sum()),
        int(df_dates['first_publish_year_source'].eq('publishDate_fallback').sum()),
        int(df_dates['first_publish_year_two_digit_inferred'].sum()),
        int(df_dates['in_project_scope_1997_2024'].sum()),
        int((~df_dates['in_project_scope_1997_2024']).sum()),
    ],
})

date_cleaning_summary


,metric,rows
0,total_rows,52478
1,first_publish_year_present,51928
2,first_publish_year_missing_after_cleaning,550
3,used_publishDate_fallback,21069
4,two_digit_year_inferred,28968
5,in_project_scope_1997_2024,37327
6,outside_project_scope_or_missing,14601


In [8]:
df_dates['first_publish_year_parse_status'].value_counts(dropna=False)


first_publish_year_parse_status
missing_firstPublishDate__parsed_text_date                      18653
two_digit_year_inferred_modern                                  15069
two_digit_year_inferred_pre_2000                                13711
missing_firstPublishDate__parsed_year_only                       1895
parsed_text_date                                                 1337
parsed_year_only                                                 1031
missing_firstPublishDate__unparseable_raw_date                    284
missing_both_date_fields                                          257
missing_firstPublishDate__two_digit_year_inferred_modern          178
missing_firstPublishDate__parsed_year_from_unrecognized_text       49
missing_firstPublishDate__two_digit_year_inferred_pre_2000         10
unparseable_raw_date                                                3
parsed_year_from_unrecognized_text                                  1
Name: count, dtype: int64

In [9]:
df_dates['first_publish_year_invalid_reason'].replace('', 'valid_or_out_of_scope_but_plausible').value_counts(dropna=False)


first_publish_year_invalid_reason
valid_or_out_of_scope_but_plausible    51928
missing_or_unparseable_year              544
before_min_valid_publication_year          6
Name: count, dtype: int64

## Missing Year Decision

Rows with missing or invalid `first_publish_year` are kept in the working dataframe but should be excluded from year/era trend analysis. They can still be used for non-temporal audits if needed.

For the project's 1997-2024 era analysis, filter with:

```python
df_scope = df_dates[df_dates['in_project_scope_1997_2024']].copy()
```


## Final Output Column

`first_publish_year` is the cleaned year column to carry forward.

In [10]:
df_dates[['bookId', 'title', 'firstPublishDate', 'publishDate', 'first_publish_year']].head(20)


,bookId,title,firstPublishDate,publishDate,first_publish_year
0,2767052-the-hunger-games,The Hunger Games,,09/14/08,2008
1,2.Harry_Potter_and_the_Order_of_the_Phoenix,Harry Potter and the Order of the Phoenix,06/21/03,09/28/04,2003
2,2657.To_Kill_a_Mockingbird,To Kill a Mockingbird,07/11/60,05/23/06,1960
3,1885.Pride_and_Prejudice,Pride and Prejudice,01/28/13,10/10/00,1913
4,41865.Twilight,Twilight,10/05/05,09/06/06,2005
5,19063.The_Book_Thief,The Book Thief,09/01/05,03/14/06,2005
6,170448.Animal_Farm,Animal Farm,08/17/45,04/28/96,1945
7,11127.The_Chronicles_of_Narnia,The Chronicles of Narnia,10/28/56,09/16/02,1956
8,30.J_R_R_Tolkien_4_Book_Boxed_Set,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,10/20/55,09/25/12,1955
9,18405.Gone_with_the_Wind,Gone with the Wind,06/30/36,04/01/99,1936
